# 03. vLLM で速くする

このノートは、[colab-oss-lab](https://github.com/moruku36/colab-oss-lab) の実験 03 です。

02 / 08 では `transformers` + `bitsandbytes`（4bit）で Gemma 4 31B を動かし、**約 6 トークン/秒** でした。
ここでは推論専用エンジン **vLLM** と、vLLM 向けの 4bit 形式（AWQ / W4A16）に替えて、どれだけ速くなるかを測ります。

| | 02 / 08（これまで） | 03（今回） |
|---|---|---|
| エンジン | transformers | **vLLM** |
| 4bit の形式 | bitsandbytes NF4（読み込み時に変換） | **AWQ W4A16（量子化済みの重みをそのまま読む）** |
| 計算の中身 | 汎用 | Marlin カーネル、CUDA Graph、PagedAttention |

試す構成:

- **A. Gemma 4 31B（AWQ 4bit）** … 02 / 08 と同じモデル。L4 22GB にぎりぎり載るかも見る
- **B. Gemma 4 26B-A4B（AWQ 4bit）** … MoE（専門家を切り替える型）。全体 26B だが1トークンあたり約 4B しか計算しないので速いはず

測るもの:

1. 1件ずつ聞いたときの速さ（トークン/秒）
2. 16件まとめて聞いたときの合計の速さ（vLLM が得意なところ）
3. 08 と同じ5問の正答数（速くしても賢さが落ちていないか）
4. VRAM と、会話の記憶（KV キャッシュ）に使える量

所要時間の目安: 30〜45 分（vLLM のインストール約5分 + 構成ごとにダウンロードと準備 各10分前後）。

---

## 実行する前に

1. **ランタイム → ランタイムのタイプを変更 → L4 GPU → Save**
2. 上から順に ▶（または「すべてのセルを実行」）
3. 終わったら **ランタイム → セッションを管理 → 解放**

## 1. GPU を確認して、vLLM を入れる

In [ ]:
import subprocess, torch
assert torch.cuda.is_available(), "ランタイムを L4 GPU にしてください"
gpu_name = torch.cuda.get_device_name(0)
vram_total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print("GPU:", gpu_name, round(vram_total_gb, 1), "GB")

In [ ]:
# vLLM は自分に合う torch をいっしょに入れる。計測は別プロセスで動かすので、再起動は不要
!pip install -q -U vllm
!python -c "import vllm, torch, transformers; print('vllm', vllm.__version__, '/ torch', torch.__version__, '/ transformers', transformers.__version__)"

## 2. 計測用のスクリプトを書く

vLLM は GPU メモリを抱えたまま手放しにくいので、**構成ごとに別プロセス**で動かします。
失敗しても次の構成に進めるようにしてあります。

In [ ]:
%%writefile bench.py
import argparse, json, re, subprocess, time

p = argparse.ArgumentParser()
p.add_argument("--name", required=True)
p.add_argument("--model", required=True)
p.add_argument("--max-model-len", type=int, default=4096)
p.add_argument("--gpu-mem", type=float, default=0.95)
p.add_argument("--eager", action="store_true")
p.add_argument("--kv-fp8", action="store_true")
a = p.parse_args()

from vllm import LLM, SamplingParams
import vllm

t0 = time.time()
kw = dict(
    model=a.model,
    max_model_len=a.max_model_len,
    gpu_memory_utilization=a.gpu_mem,
    enforce_eager=a.eager,
    language_model_only=True,   # 画像の部品を読み込まず、文章だけで使う（VRAM 節約）
    max_num_seqs=16,
)
if a.kv_fp8:
    kw["kv_cache_dtype"] = "fp8"
llm = LLM(**kw)
load_sec = time.time() - t0

SYSTEM = "You are a helpful assistant. Answer in Japanese."
QUIZ_SYSTEM = SYSTEM + " 最後の行に必ず「答え: <数字>」の形で答えだけを書いてください。"
CT = {"enable_thinking": False}

def chat(questions, max_tokens, system=SYSTEM):
    sp = SamplingParams(temperature=0, max_tokens=max_tokens)
    convs = [[{"role": "system", "content": system}, {"role": "user", "content": q}] for q in questions]
    t = time.time()
    outs = llm.chat(convs, sp, chat_template_kwargs=CT, use_tqdm=False)
    return outs, time.time() - t

SPEED_QS = [
    "小学校の児童にも分かる言葉で、GPUとVRAMの違いを3文で説明してください。",
    "量子化とは何かを、たとえ話を使って5文で説明してください。",
    "Google Colab を初めて使う人への注意点を5つ、箇条書きで書いてください。",
    "オープンソースのAIモデルを会社で使うときのメリットとデメリットを3つずつ書いてください。",
    "カレーライスの作り方を、手順ごとに番号をつけて説明してください。",
    "東京から大阪へ行く方法を3つ挙げて、それぞれの良い点を書いてください。",
    "プログラミング初心者がPythonを学ぶ順番を、6つのステップで説明してください。",
    "雨の日でも楽しめる休日の過ごし方を5つ提案してください。",
]
QUIZ = [
    ("ある数に3を足して2倍すると、その数の3倍より4小さくなります。ある数はいくつですか。", 10),
    ("1から100までの整数のうち、3でも5でも割り切れないものはいくつありますか。", 53),
    ("英単語 strawberry の中に、アルファベットの r は何個含まれていますか。", 3),
    ("A、B、C、D の4人が横一列に並びます。AとBが隣り合わない並び方は何通りですか。", 12),
    ("時計が3時15分を指しているとき、長針と短針がつくる小さいほうの角は何度ですか。", 7.5),
]

chat(["こんにちは"], 16)  # 準備運動（初回だけ遅いので計測から外す）

# 1) 1件ずつ（3問、各256トークンまで）
single = []
for q in SPEED_QS[:3]:
    outs, sec = chat([q], 256)
    n = len(outs[0].outputs[0].token_ids)
    single.append(dict(q=q, tokens=n, sec=sec, tps=n / sec, text=outs[0].outputs[0].text))
single_tps = sum(r["tokens"] for r in single) / sum(r["sec"] for r in single)

# 2) 16件まとめて
batch_qs = SPEED_QS * 2
outs, sec = chat(batch_qs, 256)
batch_tokens = sum(len(o.outputs[0].token_ids) for o in outs)
batch = dict(n=len(batch_qs), tokens=batch_tokens, sec=sec, tps=batch_tokens / sec)

# 3) 08 と同じ5問（まとめて投げる）
def extract(final):
    m = re.findall(r"答え\s*[:：]\s*\**\s*([0-9]+(?:\.[0-9]+)?)", final) or re.findall(r"([0-9]+(?:\.[0-9]+)?)", final)
    return float(m[-1]) if m else None
outs, sec = chat([q for q, _ in QUIZ], 512, system=QUIZ_SYSTEM)
quiz = []
for (q, ans), o in zip(QUIZ, outs):
    got = extract(o.outputs[0].text)
    quiz.append(dict(q=q, expected=ans, got=got, correct=got is not None and abs(got - ans) < 1e-6,
                     tokens=len(o.outputs[0].token_ids), text=o.outputs[0].text))

vram_used_mib = int(subprocess.check_output(
    ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"], text=True).split()[0])

res = dict(
    name=a.name, model=a.model, vllm=vllm.__version__,
    settings=dict(max_model_len=a.max_model_len, gpu_mem=a.gpu_mem, eager=a.eager, kv_fp8=a.kv_fp8),
    load_min=load_sec / 60, single=single, single_tps=single_tps, batch=batch,
    quiz=quiz, quiz_correct=sum(r["correct"] for r in quiz), vram_used_gb=vram_used_mib / 1024,
)
json.dump(res, open(f"/content/{a.name}.json", "w"), ensure_ascii=False, indent=1)
print("OK", a.name, f"single {single_tps:.1f} tok/s, batch {batch['tps']:.1f} tok/s, quiz {res['quiz_correct']}/5")

In [ ]:
import json, os, re, subprocess, time

def bench(name, model, attempts):
    """attempts の設定を順に試し、最初に成功したものの結果を返す"""
    for i, extra in enumerate(attempts, 1):
        label = " ".join(extra) or "標準設定"
        print(f"[{name}] 試行 {i}: {label}")
        t = time.time()
        p = subprocess.run(["python", "bench.py", "--name", name, "--model", model] + extra,
                           capture_output=True, text=True)
        log = p.stdout + p.stderr
        open(f"/content/{name}_try{i}.log", "w").write(log)
        print(f"  {'成功' if p.returncode == 0 else '失敗'}（{(time.time() - t) / 60:.1f} 分）")
        if p.returncode == 0 and os.path.exists(f"/content/{name}.json"):
            r = json.load(open(f"/content/{name}.json"))
            r["attempt"] = label
            r["tries"] = i
            m = re.search(r"GPU KV cache size: ([\d,]+) tokens", log)
            r["kv_tokens"] = int(m.group(1).replace(",", "")) if m else None
            print("  ", log.strip().splitlines()[-1])
            return r
        print("  ログの最後:")
        print("    " + "\n    ".join(log.strip().splitlines()[-12:]))
    return None

# 標準 → だめなら メモリを節約する設定 → さらに節約
ATTEMPTS = [
    [],
    ["--kv-fp8", "--max-model-len", "2048"],
    ["--kv-fp8", "--max-model-len", "1024", "--eager", "--gpu-mem", "0.97"],
]

## 3. 構成 A: Gemma 4 31B（AWQ 4bit）

使う重み: [`ebircak/gemma-4-31B-it-4bit-W4A16-AWQ`](https://huggingface.co/ebircak/gemma-4-31B-it-4bit-W4A16-AWQ)（19.2GB、Apache-2.0）

- 02 / 08 と同じ `google/gemma-4-31B-it` を、AWQ で 4bit にしたもの（vLLM 用の compressed-tensors 形式）
- 画像の部品（約1GB）は読み込まないので、GPU に載せるのは約 18GB

In [ ]:
res_a = bench("A_gemma4_31b_awq", "ebircak/gemma-4-31B-it-4bit-W4A16-AWQ", ATTEMPTS)

## 4. 構成 B: Gemma 4 26B-A4B（MoE、AWQ 4bit）

使う重み: [`cyankiwi/gemma-4-26B-A4B-it-AWQ-4bit`](https://huggingface.co/cyankiwi/gemma-4-26B-A4B-it-AWQ-4bit)（17.2GB、Apache-2.0）

- 全体は 25.2B だが、1トークンを作るときに動くのは約 3.8B だけ（128人の専門家から8人を選ぶ）
- Google の数字では 31B より少し下の賢さ（MMLU Pro 82.6% / 31B は 85.2%）

In [ ]:
res_b = bench("B_gemma4_26b_a4b_awq", "cyankiwi/gemma-4-26B-A4B-it-AWQ-4bit", ATTEMPTS)

## 5. まとめて、実行記録を出す

In [ ]:
from datetime import datetime, timezone, timedelta

# 08 の実測（transformers + bitsandbytes 4bit、Gemma 4 31B、1件ずつ、thinking オフ）
BASE = dict(name="08: transformers + bnb 4bit（31B）", single_tps=6.1, quiz_correct=5, vram_used_gb=18.2)

checks = {
    "構成 A（31B）が vLLM で L4 に載った": res_a is not None,
    "構成 A の1件ずつの速さが 08（6.1 トークン/秒）より速い": bool(res_a) and res_a["single_tps"] > BASE["single_tps"],
    "構成 A の5問の正答数が 08 と同じ（5/5）": bool(res_a) and res_a["quiz_correct"] == 5,
    "まとめて聞くと合計の速さが上がる": bool(res_a) and res_a["batch"]["tps"] > res_a["single_tps"],
}
ok = all(checks.values())

def row(r, label):
    if r is None:
        return f"| {label} | 失敗 | - | - | - | - | - |"
    return (f"| {label} | {r['single_tps']:.1f} | {r['batch']['tps']:.1f} | {r['quiz_correct']} / 5 |"
            f" {r['vram_used_gb']:.1f} GB | {r['kv_tokens'] or '-'} | {r['load_min']:.1f} 分 |")

now = datetime.now(timezone(timedelta(hours=9))).strftime("%Y-%m-%d %H:%M JST")
vv = (res_a or res_b or {}).get("vllm", "?")
L = [
    "# 実行記録: 03 vLLM で速くする",
    "",
    f"- 実行日: {now}",
    "- 実行場所: Google Colab",
    f"- GPU: {gpu_name} / VRAM {round(vram_total_gb, 1)} GB",
    f"- vLLM: {vv}",
    "- 生成設定: greedy（temperature=0）、thinking オフ、1件ずつは各256トークンまで、まとめては16件×256トークンまで",
    f"- 想定どおりか: {'はい' if ok else 'いいえ'}",
    "",
    "## まとめ",
    "",
    "| 構成 | 1件ずつ（トークン/秒） | 16件まとめて（合計トークン/秒） | 5問の正答 | VRAM | KV キャッシュ（トークン） | 準備時間 |",
    "|---|---|---|---|---|---|---|",
    f"| {BASE['name']} | {BASE['single_tps']} | - | {BASE['quiz_correct']} / 5 | {BASE['vram_used_gb']} GB | - | 約6 分 |",
    row(res_a, "A: vLLM + AWQ 4bit（31B）"),
    row(res_b, "B: vLLM + AWQ 4bit（26B-A4B MoE）"),
    "",
    "## 判定",
    "",
] + [f"- [{'x' if v else ' '}] {k}" for k, v in checks.items()] + [""]
for r, label in [(res_a, "A"), (res_b, "B")]:
    if r is None:
        continue
    L += [f"## 構成 {label}: {r['model']}", "",
          f"- 成功した設定: {r['attempt']}（{r['tries']} 回目）",
          f"- 1件ずつ: " + " / ".join(f"{x['tokens']}トークン {x['sec']:.1f}秒" for x in r["single"]),
          f"- 16件まとめて: {r['batch']['tokens']} トークン / {r['batch']['sec']:.1f} 秒",
          "- 5問: " + " ".join(f"{'○' if q['correct'] else '×'}{q['got']}" for q in r["quiz"]),
          "", "返事の例（1問目）:", "", "```", r["single"][0]["text"].strip(), "```", ""]
print("\n".join(L))

## 終わったら

**ランタイム → セッションを管理 → 解放** を必ず押してください。